# Kernel Functions and Kernel Machines
In this module, we'll take a theoretical detour into positive-definite kernels and explore how they power our first kernel machine: kernel regression. You'll discover how similarity functions enable us to solve nonlinear problems using elegant linear methods. By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

* **Positive-definite kernel**: A function $k\colon\mathbb{R}^d\times\mathbb{R}^d\to\mathbb{R}$ that measures similarity between data points such that for any sample $\{\mathbf{x}_i\}$, the resulting matrix $K_{ij}=k(\mathbf{x}_i,\mathbf{x}_j)$ is positive semidefinite. This mathematical property guarantees that $k$ corresponds to an inner product in some (possibly infinite-dimensional) feature space, allowing us to leverage all the geometric tools of Euclidean space—distances, projections, orthogonality—without ever explicitly constructing that space.

* **Kernel machine**: An algorithmic framework that replaces ordinary inner products $\langle \mathbf{x},\mathbf{x}^{\prime}\rangle$ with kernel functions $k(\mathbf{x},\mathbf{x}^{\prime})$, enabling work in an implicit high-dimensional feature space using familiar linear methods (e.g., regression, PCA, classification). By _lifting_ data through these similarity measures, we can fit linear models that exhibit complex nonlinear boundaries in the original input space.

* **Kernel ridge regression**: A nonparametric estimator that predict the label or continuous value at a test point $\mathbf{z}$ by weighting each training point $\mathbf{x}_i$ according to the similarity $k(\mathbf{z},\mathbf{x}_i)$. Points deemed more similar have greater influence on the prediction, allowing flexible modeling of smooth curves or surfaces without pre-specifying polynomial degrees or basis functions.

Ready to see how a simple similarity function unlocks powerful nonlinear models? Let's dive in!
___

## Kernel Functions
Kernel functions in machine learning are mathematical tools that enable algorithms to (implicitly) operate in _high-dimensional spaces_ without explicitly computing the coordinates in those spaces. _Huh??_

> __Kernel functions:__ A kernel function $k:\mathbb{R}^{m}\times\mathbb{R}^{m}\to\mathbb{R}$ takes vectors $\mathbf{v}_i\in\mathbb{R}^{m}$ and $\mathbf{v}_j\in\mathbb{R}^{m}$ as arguments, and computes a scalar that represents the similarity (in some sense) between the two vector arguments.  

Common kernel functions include the linear kernel: $k(\mathbf{v}_i, \mathbf{v}_j) = \mathbf{v}_i^{\top}\mathbf{v}_j$, the polynomial kernel: $k_{d}(\mathbf{v}_i, \mathbf{v}_j) = (1+\mathbf{v}_i^{\top}\mathbf{v}_j)^d$ and the radial basis function (RBF) kernel $k_{\gamma}(\mathbf{v}_i, \mathbf{v}_j) = \exp(-\gamma \lVert\mathbf{v}_i - \mathbf{v}_j\rVert_{2}^2)$ where $\gamma>0$ is a scaling factor, and $\lVert\cdot\rVert^{2}_{2}$ is the squared Euclidean norm. There are __many__ other kernel functions.

__Hmmm.__ These seem like regular functions. Where does the implicit _high-dimensional space_ part come in? Let’s explore this question.

### Implicitly high-dimensional
Kernel functions are similarity measures between points in _some high-dimensional space_. Every positive‐definite kernel $k(\mathbf{x},\mathbf{z})$ corresponds (by Mercer’s theorem) to an inner product in _some high-dimensional_ feature space:
$$
    k(\mathbf{x},\mathbf{z}) \;=\; \langle \phi(\mathbf{x}),\,\phi(\mathbf{z})\rangle_{\mathcal H},
$$
where the (nonlinear) transformation $\phi\colon\mathbb{R}^m\to\mathcal H$ embeds your original $m$-dimensional data into a (possibly infinite‐dimensional) space $\mathcal H$.  In that space, **your model is exactly linear**:

$$
\boxed{
    \hat y(\mathbf{z})
    = \phi(\mathbf{z})^\top\,\hat\theta_\lambda
    = \sum_{i=1}^n \alpha_i\,\underbrace{\langle \phi(\mathbf{z}),\,\phi(\mathbf{x}_i)\rangle}_{\text{similarity in $\mathcal H$}}
    = \sum_{i=1}^n \alpha_i\,k(\mathbf{z},\mathbf{x}_i).}
$$
So the nonlinearity lives in the $\phi(\cdot)$ transformation, but once you’re in $\mathcal H$ the regression is a perfect linear predictor. Thus, each kernel function $k(\mathbf{x},\mathbf{z})$ is _really_ a simularity between the transformed points $\phi(\mathbf{x})$ and $\phi(\mathbf{z})$ in the high-dimensional space $\mathcal H$.

Let's consider a simple example to see how this works in practice.

---

**Kernel** for $v_{i}, v_j \in \mathbb{R}$:

$$
k(v_i, v_j) \;=\; \bigl(1 + v_i\,v_j\bigr)^2.
$$

**Expansion**

$$
(1 + v_i v_j)^2
= 1 \;+\; 2\,(v_i v_j)\;+\;(v_i v_j)^2.
$$

We want to write this as a inner-product $\langle \phi(v_i), \phi(v_j)\rangle$ for some feature map $\phi\colon\mathbb{R}\to\mathbb{R}^3$ such that:
* The constant term `1` comes from the feature $\phi_0(v)=1$.
* The middle term $2\,v_i v_j$ suggests a feature proportional to $v$.
* The last term $(v_i v_j)^2$ comes from a feature proportional to $v^2$.

**Feature map**

$$
\phi(v)
=\bigl[\;1,\;\sqrt{2}\,v,\;v^2\;\bigr]^\top.
$$

Check:

$$
\langle \phi(v_i), \phi(v_j)\rangle
= 1\cdot1 \;+\; (\sqrt2\,v_i)(\sqrt2\,v_j) \;+\; (v_i^2)(v_j^2)
= 1 + 2\,v_i v_j + (v_i v_j)^2
= (1 + v_i v_j)^2\qquad\blacksquare
$$


__Wow!__. This example shows how a low-dimensional nonlinear kernel corresponds to a simple explicit—linear model in a higher-dimensional feature space! When we use a kernel function, we are essentially computing the inner product of the transformed data points in a high-dimensional space, without explicitly constructing that space. 

This is the essence of the kernel trick, a __very__ clever technique.

---

## Can any function be a kernel function?
No! Not all functions can be kernel functions:
* __Rules for a valid kernel function__: A function $k:\mathbb{R}^{m}\times\mathbb{R}^{m}\to\mathbb{R}$ is a _valid kernel function_ if and only if for any finite set of vectors $\{\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_n\in\mathbb{R}^{m}\}$, the kernel matrix $\mathbf{K}$ defined by $K_{ij} = k(\mathbf{v}_i, \mathbf{v}_j)$ satisfies the conditions:
  * The matrix $\mathbf{K}$ is symmetric: $K_{ij} = K_{ji}$.
  * The matrix $\mathbf{K}$ is positive semidefinite: for all $\boldsymbol{\alpha}\in\mathbb{R}^n$,  then $\boldsymbol{\alpha}^\top K\,\boldsymbol{\alpha}\ge0$. This is equivalent to saying that all eigenvalues of the kernel matrix $\mathbf{K}$ are non-negative (and real).

Further, when the kernel function is an (untransformed) inner product, the Kernel matrix is equal to [the Gram matrix](https://en.wikipedia.org/wiki/Gram_matrix).

* __What is the Gram matrix__? For a set of $n$-vectors, $\left\{\mathbf{v}_{1},\mathbf{v}_{2},\dots,\mathbf{v}_{n}\in\mathbb{R}^{m}\right\}$, [the Gram matrix](https://en.wikipedia.org/wiki/Gram_matrix) $\mathbf{G}\in\mathbb{R}^{n\times{n}}$ has elements $G_{ij}=\left<\mathbf{v}_{i},\mathbf{v}_{j}\right>$, where $\left<\star,\star\right>$ denotes an inner product. If the vectors $\left\{\mathbf{v}_{1},\mathbf{v}_{2},\dots,\mathbf{v}_{n}\in\mathbb{R}^{m}\right\}$ are the _columns_ of the matrix $\mathbf{X}\in\mathbb{R}^{m\times n}$, then the Gram matrix $\mathbf{G}=\mathbf{X}^\top \mathbf{X}$ is symmetric and positive semidefinite. However, if the sample vectors $\left\{\mathbf{v}_{1},\mathbf{v}_{2},\dots,\mathbf{v}_{n}\in\mathbb{R}^{m}\right\}$ are the _rows_ of the matrix $\mathbf{X}\in\mathbb{R}^{n\times m}$, then the Gram matrix $\mathbf{G}=\mathbf{X} \mathbf{X}^\top$ is symmetric and positive semidefinite.

Now that we know what a kernel function is, we can use it to define a kernel machine. Let's explore kernel regression, our first kernel machine!
___

## Kernel ridge regression
Kernel regression is a non-parametric technique to model (potentially) non-linear relationships between variables. It moves away from having a single global model to describe data and instead uses a weighted combination of multiple local (potentially nonlinear) models to represent the data.
> __Kernel ridge regression__ Kernel ridge regression uses a _kernel function_ to assign weights to new (unseen) data based on their proximity (similarity) to known data, allowing for estimating a smooth curve or function that describes the relationship between dependent and independent variables. Thus, we shift from a single global model to a weighted combination of multiple local models.

Suppose we have a dataset $\mathcal{D} = \{\left(\mathbf{x}_{i},y_{i}\right) \mid i = 1,2,\dots,n\}$, where the features $\mathbf{x}_i \in \mathbb{R}^{m}$ are $m$-dimensional vectors ($m\ll{n}$) and the target variables are continuous values $y_i \in \mathbb{R}$, e.g., the price of a house, the price of a stock, the temperature, etc. We can model this as a linear regression problem:
$$
\hat{\mathbf{y}} = \hat{\mathbf{X}}\theta + \epsilon
$$
where $\hat{\mathbf{X}}\in\mathbb{R}^{n\times{p}}$ ($p=m+1$) is an _augmented_ data matrix with the transpose of the _augmented feature vectors_ $\hat{\mathbf{x}}_{i}^{\top}$ (extra `1` appended to the features to account for the bias term) on the rows, and $\theta\in\mathbb{R}^{p}$ is an unknown parameter vector. The (regularized) least squares solution for the parameters $\theta$ is given by:
$$
\begin{equation}
\hat{\mathbf{\theta}}_{\lambda} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
\end{equation}
$$


The basic idea of kernel ridge regression is to rewrite the parameter vector $\hat{\theta}_{\lambda}$ as a sum of the _augmented feature variables_: $\hat{\theta}_{\lambda} = \sum_{i=1}^{n}\alpha_{i}\hat{\mathbf{x}}_{i}$. Then, for some (new) feature vector $\hat{\mathbf{z}}$,  the predicted output $\hat{y}$ is given by:
$$
    \hat y(\hat{\mathbf{z}})
     = \phi(\hat{\mathbf{z}})^\top\,\hat\theta_\lambda
     = \sum_{i=1}^n \alpha_i\,\langle \phi(\hat{\mathbf{z}}),\,\phi(\hat{\mathbf{x}}_i)\rangle
     = \sum_{i=1}^n \alpha_i\,k(\hat{\mathbf{z}},\hat{\mathbf{x}}_i).
$$
where $k(\hat{\mathbf{z}},\hat{\mathbf{x}}_{i})$ denotes a kernel function (similarity score) between a new (augmented) feature vector $\hat{\mathbf{z}}$ and the (known augmented) feature vector $\hat{\mathbf{x}}_{i}$.

### Expansion coefficients $\alpha$?
To get an expression for the expansion coefficients $\alpha$, we can rewrite the regularized least squares solution for the parameters $\theta_{\lambda}$. As it turns out, for any data matrix $\hat{\mathbf{X}}$, output vector $\mathbf{y}$ and regularization parameter $\lambda\geq{0}$, we can show:
$$
\begin{align*}
    \hat{\mathbf{\theta}}_{\lambda} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}\\
   \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}+\lambda\,\mathbf{I}\right)^{-1}\mathbf{y} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}\\
   \hat{\mathbf{\theta}}_{\lambda} &= \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}+\lambda\,\mathbf{I}\right)^{-1}\mathbf{y}
\end{align*}
$$
Of course, we know that the parameters $\hat{\theta}_{\lambda}$ can be expressed as a linear combination of the (augmented) expansion coefficients $\alpha$:
$$
\begin{align*}
\hat{\boldsymbol{\theta}}_{\lambda} & = \hat{\mathbf{X}}^{\top}\boldsymbol{\alpha}\\
\underbrace{\hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}+\lambda\,\mathbf{I}\right)^{-1}\mathbf{y}}_{\hat{\boldsymbol{\theta}}_{\lambda}} & = \hat{\mathbf{X}}^{\top}\boldsymbol{\alpha}\quad\Longrightarrow\text{multiply by $\hat{\mathbf{X}}$} \\
\underbrace{\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}}_{\mathbf{K}}\left(\underbrace{\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}}_{\mathbf{K}}+\lambda\,\mathbf{I}\right)^{-1}\mathbf{y} & = \underbrace{\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}}_{\mathbf{K}}\boldsymbol{\alpha}\\
\mathbf{K}\left(\mathbf{K}+\lambda\,\mathbf{I}\right)^{-1}\mathbf{y} & = \mathbf{K}\boldsymbol{\alpha}\quad\Longrightarrow\text{multiply by $\mathbf{K}^{{-1}}$}\\
\left(\mathbf{K}+\lambda\,\mathbf{I}\right)^{-1}\mathbf{y} & = \alpha\quad\blacksquare\\
\end{align*}
$$
Thus, the original parameters and the expansion coefficients $\boldsymbol{\alpha}$ are given by:
$$
\boxed{
  \boldsymbol{\alpha} \;=\; (\mathbf{K} + \lambda \mathbf{I})^{-1}\,\mathbf{y},
  \qquad
  \hat{\boldsymbol{\theta}}_\lambda \;=\; \hat{\mathbf{X}}^{\top}\boldsymbol{\alpha}.
}
$$

### Bandwidth
The bandwidth parameter $h$ in kernel regression is an important smoothing parameter; it controls how local your weights are. For example, instead of weighting by $k(\hat{\mathbf{z}},\hat{\mathbf{x}}_i)$ you use a *scaled* kernel:
$$
k_h(\hat{\mathbf{z}},\hat{\mathbf{x}}_i)
\;=\;
\frac{1}{h}\;k\Bigl(\frac{\hat{\mathbf{x}}-\hat{\mathbf{x}}_i}{h}\Bigr),
$$
where $k$ is a kernel function (e.g., Gaussian, Epanechnikov, etc.) and $h>0$ is the bandwidth parameter. Scaling by ${1}/{h}$ ensures that the kernel integrates to 1, so it remains a valid probability density function.

__How do we choose the bandwidth $h$?__ The choice of bandwidth $h$ is crucial for the performance of kernel regression:

* **Small $h$ ⇒ narrow kernel** Only points very close to $\mathbf{x}$ (within roughly distance $h$) get substantial weight. The fit follows the data very tightly (low bias, high variance).
* **Large $h$ ⇒ wide kernel** Distant points still contribute meaningfully. The fit is smoother and more stable (higher bias, lower variance).

Choosing $h$ is important: too small and you overfit noise; too large and you oversmooth real structure. In practice, one often selects $h$ by cross-validation or other data-driven methods.
___